# C1 · Too many rounds: the accuracy stays, the probabilities get ruined

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Train XGBoost with a high learning rate and deep trees for 400 rounds and observe, on the test set, **accuracy, log-loss and calibration** as a function of the rounds: the log-loss touches its minimum around 25 rounds and then rises by 45%, while the accuracy stays almost still. The training log-loss goes to almost zero.

## Dataset

`make_classification(6000, 12 features, 5 informative, flip_y=0.1)`, split 50% training, 30% test, 20% validation (`X_tr, X_te, X_val`); the model is `XGBClassifier(n_estimators=400, learning_rate=0.3, max_depth=6)`.

## Educational purpose

Understand that the overfitting of boosting shows up **in the probabilities first** and then in the classes: with `predict` (threshold 0.5) you may not notice it, but if you use the probabilities (risk, costs, ranking) they have become extreme and wrong. It is why early stopping is done on the log-loss and not on the classification error.

## Task

1. With `m.predict_proba(X, iteration_range=(0, k))` compute for `k` in `[10, 25, 50, 100, 200, 400]` the training log-loss (`train_ll`), the test log-loss (`test_ll`) and the test accuracy (`test_acc`) in a `pd.DataFrame` `table` (index `k`).
2. Save `k_min_ll` (the `k` with the minimum test log-loss), `min_ll`, `ll_400` and `ll_increase` (`ll_400 / min_ll − 1`), and `acc_change` (maximum `test_acc` minus `test_acc` at 400).
3. Save `extreme_share` (for `k = 400` and for `k = 25`, the fraction of test probabilities below 0.02 or above 0.98, in a dictionary) and `extreme_err` (for `k = 400`, among the extreme probabilities, the fraction that gets the class wrong).
4. **Cure:** train with `early_stopping_rounds=30` on `X_val, y_val` with `eval_metric="logloss"` and save `early_test_ll` and `early_round`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X, y = make_classification(6000, n_features=12, n_informative=5, class_sep=0.7, flip_y=0.1, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.5, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.6, random_state=0)
m = xgb.XGBClassifier(n_estimators=400, learning_rate=0.3, max_depth=6, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
ks = [10, 25, 50, 100, 200, 400]

## Your work

In [ ]:
table = pd.DataFrame(index=ks, columns=["train_ll", "test_ll", "test_acc"], dtype=float)
for k in ks:
    ...
table.round(3)

In [ ]:
k_min_ll = None
min_ll = None
ll_400 = None
ll_increase = None
acc_change = None

In [ ]:
extreme_share = {}
extreme_err = None

In [ ]:
early_test_ll = None
early_round = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`p = m.predict_proba(X_te, iteration_range=(0, k))[:, 1]`; `log_loss(y_te, p)`; `((p > 0.5) == y_te).mean()`. Early stopping: `xgb.XGBClassifier(n_estimators=400, learning_rate=0.3, max_depth=6, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0).fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)`; `best_iteration + 1`.'):
    check(table.notna().all().all() and table["train_ll"].is_monotonic_decreasing and table.loc[400, "train_ll"] < 0.02, "the training log-loss always drops, down to under 0.02", "iteration_range=(0, k)")
    check(k_min_ll in (10, 25, 50) and ll_increase > 0.35 and ll_400 > 0.65, "the test log-loss has its minimum between 10 and 50 rounds and at 400 it is over 35% higher (over 0.65)", "compare the rows of the table")
    check(abs(acc_change) < 0.02 and table["test_acc"].min() > 0.78, "the test accuracy varies by less than 2 points: the problem is not visible with predict", "maximum test_acc minus test_acc at 400")
    check(extreme_share[400] > 0.4 and extreme_share[25] < 0.1 and extreme_err > 0.05, "at 400 rounds over 40% of the probabilities are extreme (below 0.02 or above 0.98), at 25 less than 10%; among the extreme ones over 5% are wrong", "(p < 0.02) | (p > 0.98)")
    check(early_round < 100 and early_test_ll < 0.5 and early_test_ll < ll_400 - 0.15, "early stopping halts before 100 rounds and gives a log-loss below 0.5", "early_stopping_rounds=30 on X_val")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_too_many_rounds_probabilities_get_ruined_first.ipynb`. Open it after you pass the check, or after you have really tried.